# minRNN — Foundations (minGRU / minLSTM)

Self-contained notebook for *Were RNNs All We Needed?* (Feng et al., 2024).

minGRU and minLSTM as the linear recurrence `h_t = a_t * h_{t-1} + b_t`,
computed three ways (sequential, parallel log-space, vanilla), plus a correctness harness.

GPU on Colab: **Runtime > Change runtime type > GPU**.


## 1. Environment

In [ ]:
import torch

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device =", dev)
if dev == "cuda":
    print("gpu  =", torch.cuda.get_device_name(0))


## 2. Scan primitives

The shared recurrence engine every model reuses.

In [ ]:
import torch.nn.functional as F


# g is the positive activation that replaces tanh; log_g is its log
def g(x):
    return torch.where(x >= 0, x + 0.5, torch.sigmoid(x))


def log_g(x):
    return torch.where(x >= 0, (F.relu(x) + 0.5).log(), -F.softplus(-x))


def heinsen_associative_scan_log(log_coeffs, log_values):
    a_star = log_coeffs.cumsum(dim=1)
    log_h0_plus_b_star = torch.logcumsumexp(log_values - a_star, dim=1)
    return (a_star + log_h0_plus_b_star).exp()


def naive_sequential_scan(a, b, h0=None):
    B, T, D = a.shape
    h_prev = torch.zeros(B, D, dtype=a.dtype, device=a.device) if h0 is None else h0
    outs = []
    for t in range(T):
        h_prev = a[:, t] * h_prev + b[:, t]
        outs.append(h_prev)
    return torch.stack(outs, dim=1)


def vanilla_scan(a, b, h0=None):
    A = torch.cumprod(a, dim=1)
    inner = torch.cumsum(b / A, dim=1)
    if h0 is not None:
        inner = inner + h0.unsqueeze(1)
    return A * inner


## 3. minGRU

Gates depend on the current input, not the previous hidden state, which is what makes it parallelizable.

In [ ]:
import torch.nn as nn


def _scan_log_with_h0(log_a, log_b, h0=None):
    B, T, D = log_a.shape
    log_a = F.pad(log_a, (0, 0, 1, 0))
    if h0 is None:
        log_h0 = torch.full((B, 1, D), float("-inf"), dtype=log_b.dtype, device=log_b.device)
    else:
        log_h0 = h0.clamp(min=1e-20).log().unsqueeze(1)
    log_b = torch.cat([log_h0, log_b], dim=1)
    return heinsen_associative_scan_log(log_a, log_b)[:, 1:]


class minGRU(nn.Module):
    def __init__(self, dim, hidden_dim=None, proj_out=False):
        super().__init__()
        hidden_dim = hidden_dim or dim
        self.dim, self.hidden_dim = dim, hidden_dim
        self.to_z = nn.Linear(dim, hidden_dim, bias=False)
        self.to_h = nn.Linear(dim, hidden_dim, bias=False)
        self.to_out = nn.Linear(hidden_dim, dim, bias=False) if proj_out else nn.Identity()

    def _pre(self, x):
        return self.to_z(x), self.to_h(x)

    def forward_parallel_log(self, x, h0=None):
        pre_z, pre_h = self._pre(x)
        log_a = -F.softplus(pre_z)
        log_z = -F.softplus(-pre_z)
        log_b = log_z + log_g(pre_h)
        return self.to_out(_scan_log_with_h0(log_a, log_b, h0))

    def forward_parallel_vanilla(self, x, h0=None):
        pre_z, pre_h = self._pre(x)
        z = pre_z.sigmoid()
        return self.to_out(vanilla_scan(1.0 - z, z * g(pre_h), h0))

    def forward_sequential(self, x, h0=None):
        pre_z, pre_h = self._pre(x)
        z = pre_z.sigmoid()
        return self.to_out(naive_sequential_scan(1.0 - z, z * g(pre_h), h0))

    def forward(self, x, h0=None, mode="parallel_log"):
        return {"parallel_log": self.forward_parallel_log,
                "parallel_vanilla": self.forward_parallel_vanilla,
                "sequential": self.forward_sequential}[mode](x, h0)


## 4. minLSTM

Same structure; forget/input gates are normalized to sum to 1.

In [ ]:
class minLSTM(nn.Module):
    def __init__(self, dim, hidden_dim=None, proj_out=False):
        super().__init__()
        hidden_dim = hidden_dim or dim
        self.dim, self.hidden_dim = dim, hidden_dim
        self.to_f = nn.Linear(dim, hidden_dim, bias=False)
        self.to_i = nn.Linear(dim, hidden_dim, bias=False)
        self.to_h = nn.Linear(dim, hidden_dim, bias=False)
        self.to_out = nn.Linear(hidden_dim, dim, bias=False) if proj_out else nn.Identity()

    def _pre(self, x):
        return self.to_f(x), self.to_i(x), self.to_h(x)

    def forward_parallel_log(self, x, h0=None):
        pre_f, pre_i, pre_h = self._pre(x)
        log_f = -F.softplus(-pre_f)
        log_i = -F.softplus(-pre_i)
        log_denom = torch.logaddexp(log_f, log_i)
        log_a = log_f - log_denom
        log_b = (log_i - log_denom) + log_g(pre_h)
        return self.to_out(_scan_log_with_h0(log_a, log_b, h0))

    def _ab(self, x):
        pre_f, pre_i, pre_h = self._pre(x)
        f, i = pre_f.sigmoid(), pre_i.sigmoid()
        denom = f + i
        return f / denom, (i / denom) * g(pre_h)

    def forward_parallel_vanilla(self, x, h0=None):
        a, b = self._ab(x)
        return self.to_out(vanilla_scan(a, b, h0))

    def forward_sequential(self, x, h0=None):
        a, b = self._ab(x)
        return self.to_out(naive_sequential_scan(a, b, h0))

    def forward(self, x, h0=None, mode="parallel_log"):
        return {"parallel_log": self.forward_parallel_log,
                "parallel_vanilla": self.forward_parallel_vanilla,
                "sequential": self.forward_sequential}[mode](x, h0)


## 5. Smoke test

In [ ]:
torch.manual_seed(0)
B, T, D = 4, 64, 32
x = torch.randn(B, T, D, device=dev)

print(f"{'model':8s} {'shape':>16s} {'max|seq-log|':>14s} {'max|seq-vanilla|':>18s}")
for name, M in [("minGRU", minGRU(D)), ("minLSTM", minLSTM(D))]:
    M = M.to(dev)
    seq = M(x, mode="sequential")
    plog = M(x, mode="parallel_log")
    pvan = M(x, mode="parallel_vanilla")
    print(f"{name:8s} {str(tuple(plog.shape)):>16s} "
          f"{(seq-plog).abs().max().item():>14.2e} {(seq-pvan).abs().max().item():>18.2e}")

M = minGRU(D).to(dev)
M(x, mode="parallel_log").sum().backward()
print("grad norm (minGRU to_z):", f"{M.to_z.weight.grad.norm().item():.4f}")


## 6. Correctness harness (Foundation 2)

Float64 on CPU for tight tolerances.

In [ ]:
import itertools

MODELS = [minGRU, minLSTM]
SHAPES = [(2, 16, 8), (3, 64, 32), (1, 128, 16)]


def _model(cls, dim, seed=0):
    torch.manual_seed(seed)
    return cls(dim).double()


def test_parallel_log_matches_sequential():
    for cls, (B, T, D) in itertools.product(MODELS, SHAPES):
        M = _model(cls, D)
        x = torch.randn(B, T, D, dtype=torch.float64)
        err = (M(x, mode="sequential") - M(x, mode="parallel_log")).abs().max().item()
        assert err < 1e-8, f"{cls.__name__} {(B,T,D)} err={err:.2e}"


def test_vanilla_matches_sequential_short():
    for cls in MODELS:
        M = _model(cls, 8)
        x = torch.randn(2, 16, 8, dtype=torch.float64)
        err = (M(x, mode="sequential") - M(x, mode="parallel_vanilla")).abs().max().item()
        assert err < 1e-8, f"{cls.__name__} vanilla err={err:.2e}"


def test_gradients_finite_and_nonzero():
    for cls in MODELS:
        M = _model(cls, 16)
        x = torch.randn(2, 32, 16, dtype=torch.float64)
        M(x, mode="parallel_log").sum().backward()
        for n, p in M.named_parameters():
            assert p.grad is not None and torch.isfinite(p.grad).all() and p.grad.abs().sum() > 0, n


def test_gradcheck():
    for cls in MODELS:
        M = _model(cls, 4)
        x = torch.randn(1, 8, 4, dtype=torch.float64, requires_grad=True)
        assert torch.autograd.gradcheck(lambda i: M(i, mode="parallel_log"), (x,), eps=1e-6, atol=1e-4)


def test_overfit_tiny():
    for cls in MODELS:
        torch.manual_seed(0)
        M = cls(8, hidden_dim=64, proj_out=True)
        x, y = torch.randn(1, 10, 8), torch.randn(1, 10, 8)
        opt = torch.optim.Adam(M.parameters(), lr=3e-3)
        loss0 = None
        for step in range(1500):
            opt.zero_grad()
            loss = ((M(x, mode="parallel_log") - y) ** 2).mean()
            loss.backward()
            opt.step()
            if step == 0:
                loss0 = loss.item()
        assert loss.item() < 0.05 * loss0, f"{cls.__name__} {loss0:.4f} -> {loss.item():.4f}"


tests = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
failed = 0
for fn in tests:
    try:
        fn()
        print("PASS ", fn.__name__)
    except AssertionError as e:
        failed += 1
        print("FAIL ", fn.__name__, e)
print(f"\n{len(tests) - failed}/{len(tests)} passed")


## 7. Log-space vs vanilla stability

Vanilla underflows on long sequences; log-space stays accurate. Motivates variation 2.

In [ ]:
torch.manual_seed(0)
M = minGRU(16).to(dev)
for Tlen in [256, 1024, 4096]:
    xx = torch.randn(1, Tlen, 16, device=dev)
    seq = M(xx, mode="sequential")
    log = M(xx, mode="parallel_log")
    van = M(xx, mode="parallel_vanilla")
    print(f"T={Tlen:5d}  max|seq-log|={(seq-log).abs().max().item():.2e}   "
          f"max|seq-vanilla|={(seq-van).abs().max().item():.2e}")
